# Bias Variance Trade-Offs in RL
Julian Hsu
2025-May-01


## Problem Setup
We run an experiment where we decide the proportion p of users to assign to treatment each round. Treatment has a negative effect on reward, so the optimal policy is to assign everyone to control — but the agent does not know this upfront.

Both arms have the **same within-arm noise**. The only source of variance that depends on p is the between-arm term p·(1-p)·(μ̃_T − μ̃_C)², which is maximised at a balanced 50/50 split and vanishes at the corners (p→0 or p→1). High-λ agents are pushed toward corners to eliminate this term.

**Initialization:** The first round draws p uniformly at random from the candidate grid. With no prior data the agent has no basis to prefer either arm.

**Uncertainty via Thompson sampling:** Each round the agent draws expected arm means from their posteriors — N(x̄, σ²/n) under a flat prior — rather than using point estimates. This uncertainty means the agent cannot immediately identify control as optimal; it must accumulate enough data before its beliefs are reliable. A high-λ agent commits aggressively to whichever corner its current posterior sample favours; a low-λ agent hedges more and corrects faster when early beliefs are wrong.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


In [ ]:
def loss(reward, variance, lambda_weight):
    return -reward + lambda_weight * variance


def choose_best_allocation(treatment_outcomes, control_outcomes, p_grid, lambda_weight):
    """Pick p that minimises loss using Thompson-sampled arm means.

    Thompson sampling draws mu_t and mu_c from their posteriors rather than
    using point estimates. This gives the agent genuine uncertainty about which
    arm has higher expected reward, preventing immediate commitment to all-control
    before sufficient data has been collected.

    Posterior (flat prior + normal likelihood, known sigma):
        mu_arm ~ N(x_bar, sigma^2 / n)
    """
    n_t = len(treatment_outcomes)
    n_c = len(control_outcomes)

    # Random allocation before any arm-specific data
    if n_t < 1 or n_c < 1:
        return np.random.choice(p_grid)

    T_bar = np.mean(treatment_outcomes)
    C_bar = np.mean(control_outcomes)
    var_T = np.var(treatment_outcomes, ddof=1) if n_t > 1 else noise_treat ** 2
    var_C = np.var(control_outcomes,   ddof=1) if n_c > 1 else noise_control ** 2

    # Draw expected means from posteriors — the source of reward uncertainty
    mu_t = np.random.normal(T_bar, noise_treat  / np.sqrt(n_t))
    mu_c = np.random.normal(C_bar, noise_control / np.sqrt(n_c))

    best_loss_val = float('inf')
    best_p = p_grid[len(p_grid) // 2]
    for p in p_grid:
        reward   = p * mu_t + (1 - p) * mu_c
        # Between-arm variance uses sampled means (agent's current belief about the gap)
        variance = (p * var_T + (1 - p) * var_C
                    + p * (1 - p) * (mu_t - mu_c) ** 2)
        cur = loss(reward, variance, lambda_weight)
        if cur < best_loss_val:
            best_loss_val = cur
            best_p = p
    return best_p


In [ ]:
def outputs(lambda_weight=None):
    allocation_history = []
    cumulative_reward_history = []

    treatment_outcomes = []
    control_outcomes = []
    total_reward = 0

    for t in range(T):
        best_p = choose_best_allocation(
            treatment_outcomes, control_outcomes, p_grid, lambda_weight)

        n_treat   = min(max(int(round(n_per_round * best_p)), 1), n_per_round - 1)
        n_control = n_per_round - n_treat
        allocation_history.append(n_treat / n_per_round)

        treat_outcomes_round = true_effect + np.random.normal(0, noise_treat, n_treat)
        cont_outcomes_round  = np.random.normal(0, noise_control, n_control)

        treatment_outcomes.extend(treat_outcomes_round)
        control_outcomes.extend(cont_outcomes_round)

        total_reward += np.sum(treat_outcomes_round) + np.sum(cont_outcomes_round)
        cumulative_reward_history.append(total_reward)

    return {
        "allocations": allocation_history,
        "cumulative_rewards": cumulative_reward_history,
    }


In [4]:
import pandas as pd

In [ ]:
###### Parameters
# The agent NEVER sees these values -- it only observes realized outcomes.
true_effect   = -3     # treatment lowers the outcome; optimal policy is all-control
noise_control = 9      # both arms have equal noise
noise_treat   = 9      # equal noise means only the between-arm term varies with p

n_per_round   = 50     # users assigned each round
T             = 20     # number of rounds
p_grid        = np.linspace(0.05, 0.95, 19)   # candidate treatment proportions


In [ ]:
lambda_values = [0.0, 0.1, 0.5, 1.5]   # variance-aversion weight (0 = pure reward)
n_runs = 50

results_by_lambda = {
    lw: [outputs(lambda_weight=lw) for _ in range(n_runs)]
    for lw in lambda_values
}

def to_df(results, key):
    return pd.DataFrame([run[key] for run in results])

alloc_dfs  = {lw: to_df(results_by_lambda[lw], "allocations")       for lw in lambda_values}
cumrew_dfs = {lw: to_df(results_by_lambda[lw], "cumulative_rewards") for lw in lambda_values}


In [ ]:
rounds = np.arange(1, T + 1)
optimal_cumrew = np.zeros(T)                          # all control: E[control] = 0
worst_cumrew   = n_per_round * rounds * true_effect   # all treatment: E = true_effect per user

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Treatment Allocation ---
ax = axes[0]
for lw in lambda_values:
    m = alloc_dfs[lw].mean()
    s = alloc_dfs[lw].std()
    (line,) = ax.plot(m.values, label=f'λ = {lw}')
    ax.fill_between(range(T), (m - s).values, (m + s).values, alpha=0.15)
ax.axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='Balanced (0.5)')
ax.set_title('Mean Treatment Allocation Over Time')
ax.set_xlabel('Round')
ax.set_ylabel('Proportion Treated')
ax.legend()

# --- Cumulative Outcome ---
ax = axes[1]
ax.plot(optimal_cumrew, linestyle='--', color='black', linewidth=1.2,
        label='Optimal (all control)')
ax.plot(worst_cumrew,   linestyle=':',  color='red',   linewidth=1.2,
        label='Worst case (all treatment)')
for lw in lambda_values:
    m = cumrew_dfs[lw].mean()
    s = cumrew_dfs[lw].std()
    (line,) = ax.plot(m.values, label=f'λ = {lw}')
    ax.fill_between(range(T), (m - s).values, (m + s).values, alpha=0.15)
ax.set_title('Cumulative Outcome Over Time')
ax.set_xlabel('Round')
ax.set_ylabel('Total Reward Accumulated')
ax.legend()

plt.tight_layout()
plt.show()
